# 1분봉 Bronze–Silver 품질검사

`minute_prices_raw`의 수집 이력과 `minute_prices`의 최신 상태가
일관적인지 읽기 전용 Spark SQL로 확인합니다.

**보여주는 역량:** 데이터 품질 규칙, 업무 키 검증, Bronze/Silver 정합성,
Iceberg 메타데이터 관찰

## Goal

- Bronze와 Silver의 행 수 및 샘플을 확인합니다.
- Silver 업무 키 중복과 OHLC 오류를 검사합니다.
- Bronze 최신 버전과 Silver 값의 불일치를 찾습니다.
- Iceberg 스냅샷과 파티션 상태를 확인합니다.

## Setup

In [1]:
from pyspark.sql import SparkSession

BRONZE_TABLE = "lakehouse.bronze.minute_prices_raw"
SILVER_TABLE = "lakehouse.silver.minute_prices"

spark = (
    SparkSession.builder
    .appName("minute-price-quality-checks")
    .getOrCreate()
)
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sparkContext.setLogLevel("ERROR")

## Data

### 1. 행 수와 최신 샘플

In [2]:
spark.sql(f"""
SELECT 'bronze' AS layer, COUNT(*) AS row_count
FROM {BRONZE_TABLE}
UNION ALL
SELECT 'silver', COUNT(*)
FROM {SILVER_TABLE}
""").show(truncate=False)

spark.sql(f"""
SELECT *
FROM {SILVER_TABLE}
ORDER BY candle_time DESC, symbol
LIMIT 10
""").show(truncate=False)

+------+---------+
|layer |row_count|
+------+---------+
|bronze|390      |
|silver|390      |
+------+---------+



+------+-------------------+--------+-----------------+------------------+------------------+------------------+------+--------+--------------------------+
|symbol|candle_time        |interval|open             |high              |low               |close             |volume|source  |collected_at              |
+------+-------------------+--------+-----------------+------------------+------------------+------------------+------+--------+--------------------------+
|AAPL  |2026-07-17 19:59:00|1m      |334.1199951171875|334.2300109863281 |333.46051025390625|333.739990234375  |919337|yfinance|2026-07-18 11:55:26.899268|
|AAPL  |2026-07-17 19:58:00|1m      |333.9200134277344|334.19000244140625|333.70001220703125|334.1400146484375 |540586|yfinance|2026-07-18 11:55:26.899268|
|AAPL  |2026-07-17 19:57:00|1m      |333.75           |334.0799865722656 |333.6600036621094 |333.94000244140625|403387|yfinance|2026-07-18 11:55:26.899268|
|AAPL  |2026-07-17 19:56:00|1m      |333.8900146484375|334.23999

## Results

### 2. Silver 업무 키와 OHLC 규칙

In [3]:
spark.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT struct(
        symbol, interval, candle_time, source
    )) AS distinct_business_keys,
    SUM(CASE
        WHEN high < greatest(open, close, low)
          OR low > least(open, close, high)
        THEN 1 ELSE 0
    END) AS invalid_ohlc_rows,
    SUM(CASE
        WHEN volume < 0
        THEN 1 ELSE 0
    END) AS negative_volume_rows
FROM {SILVER_TABLE}
""").show(truncate=False)

+----------+----------------------+-----------------+--------------------+
|total_rows|distinct_business_keys|invalid_ohlc_rows|negative_volume_rows|
+----------+----------------------+-----------------+--------------------+
|390       |390                   |0                |0                   |
+----------+----------------------+-----------------+--------------------+



### 3. Bronze 재수집 버전 확인

In [4]:
spark.sql(f"""
SELECT
    symbol,
    interval,
    candle_time,
    source,
    COUNT(*) AS collected_versions,
    MIN(collected_at) AS first_collected_at,
    MAX(collected_at) AS last_collected_at
FROM {BRONZE_TABLE}
GROUP BY symbol, interval, candle_time, source
HAVING COUNT(*) > 1
ORDER BY collected_versions DESC, candle_time DESC
LIMIT 10
""").show(truncate=False)

+------+--------+-----------+------+------------------+------------------+-----------------+
|symbol|interval|candle_time|source|collected_versions|first_collected_at|last_collected_at|
+------+--------+-----------+------+------------------+------------------+-----------------+
+------+--------+-----------+------+------------------+------------------+-----------------+



### 4. Bronze 최신값과 Silver 정합성

In [5]:
spark.sql(f"""
WITH latest_bronze AS (
    SELECT
        symbol,
        interval,
        candle_time,
        source,
        open,
        high,
        low,
        close,
        volume,
        collected_at
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY
                    symbol, interval, candle_time, source
                ORDER BY collected_at DESC
            ) AS row_num
        FROM {BRONZE_TABLE}
    )
    WHERE row_num = 1
)
SELECT
    COUNT(*) AS mismatched_rows
FROM latest_bronze AS bronze
FULL OUTER JOIN {SILVER_TABLE} AS silver
  ON  bronze.symbol = silver.symbol
  AND bronze.interval = silver.interval
  AND bronze.candle_time = silver.candle_time
  AND bronze.source = silver.source
WHERE bronze.symbol IS NULL
   OR silver.symbol IS NULL
   OR NOT (
       bronze.open <=> silver.open
       AND bronze.high <=> silver.high
       AND bronze.low <=> silver.low
       AND bronze.close <=> silver.close
       AND bronze.volume <=> silver.volume
       AND bronze.collected_at <=> silver.collected_at
   )
""").show(truncate=False)

+---------------+
|mismatched_rows|
+---------------+
|0              |
+---------------+



### 5. Iceberg 메타데이터 확인

In [6]:
spark.sql(f"""
SELECT
    committed_at,
    snapshot_id,
    operation,
    summary['added-records'] AS added_records,
    summary['total-records'] AS total_records,
    summary['total-data-files'] AS total_data_files
FROM {SILVER_TABLE}.snapshots
ORDER BY committed_at DESC
LIMIT 10
""").show(truncate=False)

spark.sql(f"""
SELECT
    partition,
    record_count,
    file_count,
    total_data_file_size_in_bytes
FROM {SILVER_TABLE}.partitions
""").show(truncate=False)

+-----------------------+-------------------+---------+-------------+-------------+----------------+
|committed_at           |snapshot_id        |operation|added_records|total_records|total_data_files|
+-----------------------+-------------------+---------+-------------+-------------+----------------+
|2026-07-18 12:11:09.316|4578971807099254810|append   |390          |390          |1               |
|2026-07-18 11:44:39.799|3707086788271620679|overwrite|NULL         |0            |0               |
+-----------------------+-------------------+---------+-------------+-------------+----------------+

+------------+------------+----------+-----------------------------+
|partition   |record_count|file_count|total_data_file_size_in_bytes|
+------------+------------+----------+-----------------------------+
|{2026-07-17}|390         |1         |10812                        |
+------------+------------+----------+-----------------------------+



## Takeaways

- `distinct_business_keys == total_rows`이면 Silver 중복이 없습니다.
- `invalid_ohlc_rows == 0`과 `negative_volume_rows == 0`이 기대값입니다.
- `mismatched_rows == 0`이면 Bronze 최신값과 Silver가 일치합니다.
- 불일치가 있으면 메달리온 파이프라인의 `MERGE INTO`를 다시 실행하고
  해당 수집 `run_id`를 조사합니다.